In [ ]:
import yaml
import pandas as pd
import digitalhub as dh
from digitalhub import get_model

from data_preparation import *
from modelling import *

# Project creation

In [ ]:
project = dh.get_or_create_project("fair-hiring-tabular2")
with open("../../metadata/aipc_dh.yaml", "r") as metadata_spec:
    aipc_dh_spec = yaml.safe_load(metadata_spec)

In [ ]:
project

# Data preparation stage

## Load dataset

In [ ]:
config = {"data_name": "recruitmentdataset.csv", "url": "https://raw.githubusercontent.com/AlbanaCelepija/enhanced_mlops/refs/heads/main/framework/library/use_cases/tabular/src/local_platform/artifacts/data/recruitmentdataset-2022.csv"}
data_preparation_fn = project.new_function(name="load_data",
                                   kind="python",
                                   python_version="PYTHON3_10",
                                   code_src="dh_data_preparation.py",
                                   handler="load_data_real",
                                   requirements=["scikit-learn"],
                                   labels=["data_preprocessing", "baseline"],)
data_preparation_fn = data_preparation_fn.run("job", parameters=config, wait=True)

In [ ]:
raw_training_di = project.get_dataitem(config["data_name"])
raw_training_df = raw_training_di.as_df()
raw_training_df

## Data profiling

In [ ]:
config = {"data_name": "recruitmentdataset.csv", "url": "https://raw.githubusercontent.com/AlbanaCelepija/enhanced_mlops/refs/heads/main/framework/library/use_cases/tabular/src/local_platform/artifacts/data/recruitmentdataset-2022.csv"}
data_preparation_fn = project.new_function(name="load_data",
                                   kind="python",
                                   python_version="PYTHON3_10",
                                   code_src="dh_data_preparation.py",
                                   handler="load_data_real",
                                   requirements=["scikit-learn"],
                                   labels=["data_preprocessing", "baseline"],)
data_preparation_fn = data_preparation_fn.run("job", parameters=config, wait=True)

## Split train-validate-test set

In [ ]:
config = {"test_size": 0.2, "valid_size": 0.25, "random_state": 42} # 0.25 x 0.8 = 0.2

split_fn = project.new_function(
    name="split-train-valid-test",
    kind="python",
    python_version="PYTHON3_10",
    code_src="dh_data_preparation.py",
    handler="split_train_valid_test_data_real",
    requirements=["scikit-learn", "numpy<2"],
    labels=["data_preprocessing", "baseline"]
)
split_run = split_fn.run(action="job", inputs={"data": raw_training_di.key}, parameters=config, wait=True)


## Preprocess Training data

In [ ]:
training_di = project.get_dataitem('training_set_X')
valid_di = project.get_dataitem('validation_set_X')
test_di = project.get_dataitem('test_set_X')
boolean_features = "ind-debateclub,ind-programming_exp,ind-international_exp,ind-entrepeneur_exp,ind-exact_study,decision"
categorical_features = "sport,ind-degree,company"
config_train = {"di_name": "train_data", "boolean_features": boolean_features, "categorical_features": categorical_features}
config_valid = {"di_name": "valid_data", "boolean_features": boolean_features, "categorical_features": categorical_features}
config_test = {"di_name": "test_data", "boolean_features": boolean_features, "categorical_features": categorical_features}

preprocess_fn = project.new_function(
    name="preprocess_train_data",
    kind="python",
    python_version="PYTHON3_10",
    code_src="dh_data_preparation.py",
    handler="preprocess_train_data_real",
    requirements=["scikit-learn", "numpy<2"],
    labels=["data_preprocessing", "baseline"]
)
preprocess_fn.run(action="job", inputs={"training_di": training_di.key}, parameters=config_train, wait=True)
preprocess_fn.run(action="job", inputs={"training_di": valid_di.key}, parameters=config_valid, wait=True)
preprocess_fn.run(action="job", inputs={"training_di": test_di.key}, parameters=config_test, wait=True)

# Modelling stage

In [ ]:
config={"sensitive_features": ["nationality", "gender"], "random_state":42, "id_feature": "Id", "target_feature": "decision"}
train_fn = project.new_function(
    name="train-classifier",
    kind="python",
    python_version="PYTHON3_10",
    code_src="dh_modelling.py",
    handler="train_model_real",
    requirements=["scikit-learn", "numpy<2"],
    labels=["model_training", "baseline"],
)
train_ds = project.get_dataitem('train_data')
train_run = train_fn.run(action="job", 
                         inputs={"data_train": train_ds.key},
                         parameters=config, 
                         wait=True
                        )

In [ ]:
model = train_run.output("model")
#model = project.get_model("hiring_classifier")
model


## Evaluate overall performance metrics

In [ ]:
config={"sensitive_features": ["nationality", "gender"], "id_feature": "Id", "target_feature": "decision"}
valid_ds = project.get_dataitem('valid_data')

model_evaluation_accuracy_overall_fn = project.new_function(
    name="model-evaluation-accuracy-overall",
    kind="python",
    python_version="PYTHON3_10",
    code_src="dh_modelling.py",
    handler="model_evaluation_accuracy_overall_real",
    requirements=["scikit-learn", "numpy<2"],
    labels=["model_evaluation", "baseline"],
)
model_evaluation_accuracy_overall_run = model_evaluation_accuracy_overall_fn.run(
    action="job",
    inputs={"model": model.key, "data_valid": valid_ds.key}, 
    parameters=config, 
    wait=True
)

## Evaluate performance metrics for each demographic group

In [ ]:
config={"sensitive_features": ["nationality", "gender"], "id_feature": "Id", "target_feature": "decision"}
valid_ds = project.get_dataitem('valid_data')

model_evaluation_accuracy_demographic_groups_fn = project.new_function(
    name="model-evaluation-accuracy-demographic-groups",
    kind="python",
    python_version="PYTHON3_10",
    code_src="dh_modelling.py",
    handler="model_evaluation_accuracy_demographic_groups_real",
    requirements=["scikit-learn", "numpy<2"],
    labels=["model_evaluation", "baseline"],
)
model_evaluation_accuracy_demographic_groups_run = model_evaluation_accuracy_demographic_groups_fn.run(
    action="job",
    inputs={"model": model.key, "data_valid": valid_ds.key}, 
    parameters=config, 
    wait=True
)

# Operationalisation stage

In [ ]:
serve_func = project.new_function(
    name="serve-classifier",
    kind="sklearnserve",
    path=model.key,
)
serve_run = serve_func.run("serve", wait=True)
serve_run

# Inference 

In [ ]:
test_df = data_preparation_fn.output("dataset").as_df().head()
test_df = test_df.drop(columns=["decision", "Id", "gender", "nationality"])
test_df


In [ ]:
data = test_df.values.tolist()
json_payload = {"inputs": [{"name": "input-0", "shape": [5, 23], "datatype": "FP32", "data": data}]}
json_payload

In [ ]:
result = serve_run.refresh().invoke(json=json_payload).json()
print("Prediction result:")
print(result)

# Workflow orchestration

## Baseline Requirements Dimension 

In [ ]:
workflow_baseline = project.new_workflow(
    name="baseline-pipeline",
    kind="hera",
    code_src="pipeline.py",
    handler="baseline_pipeline",
)

In [ ]:
wf_build = workflow_baseline.run("build", wait=True)
wf_run = workflow_baseline.run("pipeline", wait=True)

## Fairness Requirements Dimension

In [ ]:
workflow_fairness = project.new_workflow(
    name="baseline-pipeline",
    kind="hera",
    code_src="pipeline.py",
    handler="fairness_pipeline",
)

In [ ]:
wf_build = workflow_fairness.run("build", wait=True)
wf_run = workflow_fairness.run("pipeline", wait=True)